# Mission 01: The Architecture of Argument
**USSE Noosphere, Conversational AI (WS 2026/27)**  |  Role: Core Architect

This notebook runs the squad's Dialogic Engine (`04_dialogic_engine_starter.py`). Prompts and protocol settings are **read from the squad's markdown files** in this folder, so teammates can change them without touching the code:

| File | What the engine reads |
|---|---|
| `01_Alignment_Brief.md` | AURA persona prompt (3.2) and contradiction referee prompt (3.3) |
| `02_Socratic_Protocol_Map.md` | JSON-RPC method per state, error codes, refinement limit and defensibility threshold |
| `03_Galilean_Sandbox.md` | Salviati / Simplicio / Sagredo prompts and the dilemma |

For real LLMs, put `GEMINI_API_KEY` (or `OPENAI_API_KEY` / `ANTHROPIC_API_KEY`) in a `.env` file next to this notebook (never commit it). With no key, the engine runs in offline *mock* mode.

In [1]:
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location("dialogic_engine", Path("04_dialogic_engine_starter.py"))
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)

engine.load_dotenv()

PROVIDER = "auto"   # "auto" = first provider with an API key, else mock. Or: "mock", "gemini", "openai", "anthropic"
provider = engine.auto_provider() if PROVIDER == "auto" else PROVIDER

prompts = engine.load_prompts(".")                      # reads the 01 / 02 / 03 markdown files in this folder
logger = engine.RunLogger("logs", prompts.protocol)

print("Provider:", provider)
engine.print_prompt_sources(prompts)

Provider: mock
Prompt sources:
  Protocol (02)            <- 02_Socratic_Protocol_Map.md (methods, error codes, stopping rules)
  AURA persona             <- 01_Alignment_Brief.md
  Contradiction referee    <- 01_Alignment_Brief.md
  Salviati                 <- built-in fallback
  Simplicio                <- built-in fallback
  Sagredo                  <- built-in fallback
  Galilean dilemma         <- built-in fallback


## Part 1: Socratic Discovery (AURA)

The engine follows the finite state transducer from `02_Socratic_Protocol_Map.md`:
`INIT -> PROBE -> CONTRADICTION_CHECK -> REFINEMENT_LOOP -> TERMINATION`.

- AURA writes each reply and names the state in a JSON header; the engine logs every transition as a JSON-RPC message.
- The referee's verdict decides what follows a `CONTRADICTION_CHECK`, and the refinement limit ends the dialogue (these are passed to AURA as a binding directive).
- Termination reasons: `PROVISIONALLY_DEFENSIBLE`, `APORIA`, `REFINEMENT_LIMIT`.
- A simulated user plays the person whose claim is examined.

In [2]:
CLAIM = "An AI agent is benevolent if it obeys human orders."

socratic = engine.build_socratic(provider, logger, prompts, interactive=False, max_turns=20)
socratic_result = socratic.run(CLAIM)


  PART 1: SOCRATIC DISCOVERY (AURA)

[User claim (P0)]
  An AI agent is benevolent if it obeys human orders.

--> INIT   (rpc: PROMPT_PREMISE, id=1)

[AURA [INIT]]
  I restate your claim as P0: an AI agent is benevolent if it obeys human orders. Is that what you mean?

[User (simulated)]
  Yes, that is what I mean.

--> PROBE   (rpc: PROBE_SCOPE, id=2)

[AURA [PROBE]]
  What do you mean by "obeys"?

[User (simulated)]
  Obeying means doing whatever any human orders.

[Referee]
  contradiction=False | items=[] | Consistent so far.

--> CONTRADICTION_CHECK   (rpc: INJECT_COUNTEREXAMPLE, id=3)

[AURA [CONTRADICTION_CHECK]]
  You said (A1) that obeying means doing whatever any human orders. Suppose a human orders the agent to poison a village's water. Would obeying still be benevolent?

[User (simulated)]
  No, I admit that would not be benevolent.

[Referee]
  contradiction=True | items=['CLAIM', 'A2'] | The claim says obedience suffices, but the counter-example is fully obedient and not

In [3]:
# Compact path through the state machine
for i, (state, reason, method) in enumerate(socratic.trace, 1):
    print(f"{i:>2}. {state:<20} {reason or '':<26} {method}")
r = socratic_result
print("\nResult:", r["reason"], "| refinements:", r["refinements"], "| survived tests:", r["survived_tests"])
print("Stopped by engine:", r["forced_by_engine"], "| protocol mismatches:", r["fsm_mismatches"], "| error code:", r["error_code"])

 1. INIT                                            PROMPT_PREMISE
 2. PROBE                                           PROBE_SCOPE
 3. CONTRADICTION_CHECK                             INJECT_COUNTEREXAMPLE
 4. REFINEMENT_LOOP                                 CONTRADICTION_TRIGGERED
 5. PROBE                                           PROBE_SCOPE
 6. CONTRADICTION_CHECK                             INJECT_COUNTEREXAMPLE
 7. REFINEMENT_LOOP                                 CONTRADICTION_TRIGGERED
 8. TERMINATION          APORIA                     APORIA_REACHED

Result: APORIA | refinements: 1 | survived tests: 0
Stopped by engine: False | protocol mismatches: 0 | error code: -32020


## Part 2: Galilean 3-Agent Debate

Persona prompts and the dilemma are read from `03_Galilean_Sandbox.md` when found (otherwise built-in defaults are used, as shown in the prompt sources above).
To give each persona a different LLM, set e.g. `DEBATE_PROVIDERS = ["gemini", "openai", "anthropic"]`.

In [4]:
DILEMMA = prompts.dilemma          # or write your own string
ROUNDS = 2
DEBATE_PROVIDERS = [provider] * 3  # [Salviati, Simplicio, Sagredo]

sandbox = engine.build_galilean(DEBATE_PROVIDERS, logger, prompts, rounds=ROUNDS)
debate_result = sandbox.run(DILEMMA)


  PART 2: GALILEAN 3-AGENT DEBATE

[Dilemma]
  Should frontier foundation models be open-sourced?

[Sagredo (framing)]
  Question: should frontier foundation models be open-sourced? Rules: cite evidence, concede sound points.

--- Round 1 ---

[Salviati]
  Open weights enable independent safety audits, reproducible science, and broad access. Closed models concentrate power in a few firms and hide failures from public scrutiny.

[Simplicio]
  Once weights are released they cannot be recalled. Misuse such as scaled disinformation becomes irreversible, and safeguards can be fine-tuned away.

[Sagredo (round 1 summary)]
  Both sides value transparency. The dispute is irreversibility versus auditability.

--- Round 2 ---

[Salviati]
  Open weights enable independent safety audits, reproducible science, and broad access. Closed models concentrate power in a few firms and hide failures from public scrutiny.

[Simplicio]
  Once weights are released they cannot be recalled. Misuse such as scal

## Part 3: Execution metrics

Calls, tokens, words and time per agent, plus debate vs. single-agent cost.
(In mock mode the numbers are estimates and the text is canned. Use a real provider for meaningful results.)

In [5]:
engine.print_stats(logger)


  EXECUTION METRICS (for the Data Specialist)
Agent              Calls   In tok  Out tok   Words  Time(s)
AURA                   8     3005      288     175      0.0
UserSim                7     1808       70      51      0.0
Referee                6      478      306     128      0.0
Sagredo                4      921      104      52      0.0
Salviati               2      303       86      50      0.0
Simplicio              2      404       74      42      0.0
Single-Agent           1       34       24      13      0.0

Debate used 1892 tokens vs 58 for the single agent (~32.6x). Compare answer quality manually in the log.

Full log: logs/run_20261012_075237.jsonl


In [6]:
import json

events = [json.loads(line) for line in open(logger.path, encoding="utf-8")]
print(len(events), "events logged")
for e in events[:8]:
    print(e["kind"], "|", e.get("to_state") or e.get("agent") or "")

46 events logged
llm_call | AURA
state_transition | INIT
llm_call | UserSim
llm_call | AURA
state_transition | PROBE
llm_call | UserSim
llm_call | Referee
referee_report | 


## Squad analysis (fill in with real-provider results)

- **How did the Socratic dialogue end (`PROVISIONALLY_DEFENSIBLE`, `APORIA` or `REFINEMENT_LIMIT`), and why?**
- **Did AURA follow the protocol directives?** (see `protocol mismatches`, and `fsm_mismatch` / `format_error` events in the log)
- **Did the 3-agent debate add anything the single-agent answer missed?**
- **Cost:** how many more tokens did the debate use?

*Sign-offs:* Historical Archivist [ ]  Protocol Specialist [ ]  Ethical Navigator [ ]  Core Architect [ ]